# 🤖 Pipeline de Modélisation Prédictive du Rendement Agricole

Ce notebook implémente l'entraînement, l'évaluation et la comparaison des modèles de Machine Learning (Régression Ridge, Random Forest et XGBoost) pour prédire le rendement agricole (`log1p_Rendement`).

Il intègre l'ensemble des caractéristiques physiques de la planète (Couche 1), les intrants agricoles, et les variables socio-démographiques, tout en évitant le data leakage grâce à une division temporelle stricte.

In [ ]:
# ── 1. Imports et Configuration ──────────────────────────────────────────
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

# Style des graphiques
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

DATA_PATH = "data/cleaned/dataset_final_modelisation.csv"
MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

## 📂 2. Chargement des données

In [ ]:
print(f"📂 Chargement du dataset : {DATA_PATH}...")
df = pd.read_csv(DATA_PATH)
print(f"✅ Dataset chargé. Dimensions : {df.shape[0]:,} lignes, {df.shape[1]} colonnes")

## 🛠️ 3. Préparation des variables (Features et Target)

Conformément aux consignes :
- Les colonnes d'identification `['Pays_EN', 'Produit', 'ISO']` sont conservées dans le DataFrame mais exclues de l'entraînement.
- La cible est `log1p_Rendement` (avec évaluation finale convertie sur le rendement réel en kg/ha).
- Les features sont l'ensemble des autres colonnes numériques (incluant les variables physiques, géologiques, climatiques, et les variables one-hot encodées).

In [ ]:
# Définition de la cible et des identifiants
target_col = 'log1p_Rendement'
raw_target_col = 'Rendement_kgha'
id_cols = ['ISO', 'Produit', 'Pays_EN']

# Identification automatique des features numériques d'entraînement
feature_cols = [col for col in df.columns if col not in [target_col, raw_target_col] + id_cols]

print(f"🎯 Cible : {target_col}")
print(f"🔑 Identifiants : {id_cols}")
print(f"📊 Nombre de features d'entraînement X : {len(feature_cols)}")

## ⏳ 4. Division Temporelle des Données (Split Train/Test)

Pour simuler un scénario de prédiction dans le futur et éviter le *data leakage* temporel :
- **Train set** : Données jusqu'en 2013 incluse.
- **Test set** : Données à partir de 2014.

In [ ]:
df_train = df[df['Annee'] <= 2013].copy()
df_test = df[df['Annee'] > 2013].copy()

X_train = df_train[feature_cols]
y_train = df_train[target_col]

X_test = df_test[feature_cols]
y_test = df_test[target_col]

print(f"📈 Train set (<= 2013) : {X_train.shape[0]:,} lignes")
print(f"📉 Test set  (> 2013)  : {X_test.shape[0]:,} lignes")

## ⚙️ 5. Construction du Pipeline de Preprocessing

Nous appliquons :
1. Une **imputation par la médiane** pour boucher les valeurs manquantes (notamment sur les intrants ou certaines variables socio-économiques).
2. Une **standardisation** pour mettre à l'échelle toutes les variables physiques et géographiques.

In [ ]:
# Preprocessor pour variables numériques uniquement
preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), feature_cols)
 stream])

## 🤖 6. Entraînement et Évaluation des Modèles

In [ ]:
# Dictionnaire pour stocker les résultats de performance
model_results = {}

# Définition des modèles
models = {
    "Régression Ridge": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1),
    "XGBoost": XGBRegressor(n_estimators=120, max_depth=6, learning_rate=0.08, random_state=42, n_jobs=-1)
}

for name, model in models.items():
    print(f"\n[INFO] Entraînement du modèle : {name}...")
    
    # Création du pipeline complet
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    # Entraînement
    pipeline.fit(X_train, y_train)
    
    # Prédictions sur l'échelle log
    y_pred_train_log = pipeline.predict(X_train)
    y_pred_test_log = pipeline.predict(X_test)
    
    # Reconversion vers l'échelle d'origine (Rendement réel en kg/ha)
    y_train_orig = np.expm1(y_train)
    y_test_orig = np.expm1(y_test)
    y_pred_train_orig = np.expm1(y_pred_train_log)
    y_pred_test_orig = np.expm1(y_pred_test_log)
    
    # Calcul des métriques réelles
    r2_train = r2_score(y_train_orig, y_pred_train_orig)
    r2_test = r2_score(y_test_orig, y_pred_test_orig)
    mae_train = mean_absolute_error(y_train_orig, y_pred_train_orig)
    mae_test = mean_absolute_error(y_test_orig, y_pred_test_orig)
    rmse_train = root_mean_squared_error(y_train_orig, y_pred_train_orig)
    rmse_test = root_mean_squared_error(y_test_orig, y_pred_test_orig)
    
    print(f"✅ {name} terminé.")
    print(f"   Train R2 : {r2_train:.4f} | Test R2 : {r2_test:.4f}")
    print(f"   Test MAE : {mae_test:.2f} kg/ha | Test RMSE : {rmse_test:.2f} kg/ha")
    
    # Sauvegarde des résultats et du modèle
    model_results[name] = {
        'pipeline': pipeline,
        'r2_train': r2_train,
        'r2_test': r2_test,
        'mae_test': mae_test,
        'rmse_test': rmse_test,
        'y_pred_test_orig': y_pred_test_orig
    }

## 📊 7. Synthèse et Comparaison des Modèles

In [ ]:
# Construction du tableau récapitulatif des métriques
df_metrics = pd.DataFrame({
    name: {
        "Train R²": f"{res['r2_train']:.4f}",
        "Test R²": f"{res['r2_test']:.4f}",
        "Test MAE (kg/ha)": f"{res['mae_test']:,.2f}",
        "Test RMSE (kg/ha)": f"{res['rmse_test']:,.2f}"
    } for name, res in model_results.items()
}).T

import sys
from IPython.display import display
display(df_metrics)

## 📈 8. Visualisations et Diagnostics

In [ ]:
# 1. Graphique des valeurs réelles vs prédictions pour le meilleur modèle (Ridge/XGBoost)
best_model_name = df_metrics["Test R²"].astype(float).idxmax()
best_res = model_results[best_model_name]
y_test_orig = np.expm1(y_test)

plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test_orig, y=best_res['y_pred_test_orig'], alpha=0.3, color='steelblue')
plt.plot([0, y_test_orig.max()], [0, y_test_orig.max()], 'r--', lw=2, label="Parfaite prédiction")
plt.title(f"📊 Rendement Réel vs Prédit (Meilleur modèle : {best_model_name})", fontsize=14, weight='bold')
plt.xlabel("Rendement Réel (kg/ha)")
plt.ylabel("Rendement Prédit (kg/ha)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 2. Importance des caractéristiques pour XGBoost
xgb_pipeline = model_results["XGBoost"]['pipeline']
xgb_model = xgb_pipeline.named_steps['model']

# Récupérer l'importance des variables
importances = xgb_model.feature_importances_
df_imp = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': importances
}).sort_values('Importance', ascending=False).head(20)

plt.figure(figsize=(12, 8))
sns.barplot(data=df_imp, x='Importance', y='Feature', palette='viridis')
plt.title("🏆 Top 20 des Caractéristiques Clés (XGBoost)", fontsize=14, weight='bold')
plt.xlabel("Importance relative")
plt.ylabel("Variables")
plt.tight_layout()
plt.show()

## 💾 9. Sauvegarde du Meilleur Modèle

Le meilleur modèle est sauvegardé sous forme de pipeline complet (comprenant l'imputation et le scaling) dans le dossier `models` pour être utilisé dans le moteur de simulation de la colonie.

In [ ]:
best_pipeline = best_res['pipeline']
model_save_path = os.path.join(MODEL_DIR, "best_model_pipeline.joblib")
joblib.dump(best_pipeline, model_save_path)
print(f"💾 Pipeline du meilleur modèle ({best_model_name}) sauvegardé dans : {model_save_path}")

# Sauvegarder un échantillon de test pour validation
test_sample = df_test.sample(min(1000, len(df_test)), random_state=42)
test_sample.to_csv("data/cleaned/test_sample.csv", index=False)
print("💾 Échantillon de test de 1000 lignes sauvegardé dans : data/cleaned/test_sample.csv")